# Réseau de Neurones Profond pour la Classification d'Images : Application

Préparé avec ❤️ par <a href="https://linkedin.com/in/dany-anderson-guimefack">Dany Anderson </a> <br>
Email📧: dany.guimefack@centrale-casablanca.ma

Dans ce notebook, vous utiliserez les fonctions implémentées dans le notebook précédent pour construire un réseau de neurones profond et l'appliquer à la classification d'images de chats.

**À la fin de ce workshop, vous serez capable de :**

- Construire et entraîner un réseau de neurones profond à L couches, et l'appliquer à l'apprentissage supervisé

**Note sur l'évaluateur automatique :** Ce notebook contient des cellules évaluées automatiquement. Évitez d'ajouter des `print` supplémentaires, de modifier les paramètres des fonctions, ou d'utiliser des variables globales dans les exercices.

Commençons !

## Table des matières
- [1 - Packages](#1)
- [2 - Chargement et traitement des données](#2)
- [3 - Architecture du modèle](#3)
 - [3.1 - Réseau de neurones à 2 couches](#3-1)
 - [3.2 - Réseau de neurones profond à L couches](#3-2)
 - [3.3 - Méthodologie générale](#3-3)
- [4 - Réseau de neurones à deux couches](#4)
 - [Exercice 1 - two_layer_model](#ex-1)
 - [4.1 - Entraîner le modèle](#4-1)
- [5 - Réseau de neurones à L couches](#5)
 - [Exercice 2 - L_layer_model](#ex-2)
 - [5.1 - Entraîner le modèle](#5-1)
- [6 - Analyse des résultats](#6)
- [7 - Testez avec votre propre image (exercice optionnel)](#7)

<a name='1'></a>
## 1 - Packages

Commencez par importer tous les packages dont vous aurez besoin dans ce notebook.

- [numpy](https://www.numpy.org/) est le package fondamental pour le calcul scientifique avec Python.
- [matplotlib](http://matplotlib.org) est une bibliothèque pour tracer des graphiques en Python.
- [h5py](http://www.h5py.org) est un package courant pour interagir avec des données stockées au format H5.
- [PIL](http://www.pythonware.com/products/pil/) et [scipy](https://www.scipy.org/) sont utilisés pour tester votre modèle avec votre propre image à la fin.
- `dnn_app_utils` fournit les fonctions implémentées dans le notebook précédent.
- `np.random.seed(1)` est utilisé pour assurer la reproductibilité des résultats. Ne le modifiez pas.

In [ ]:
### v1.1

In [ ]:
import time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy
from PIL import Image
from scipy import ndimage
from dnn_app_utils_v3 import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - Chargement et traitement des données

Vous utiliserez le même jeu de données "Chat / non-Chat" que dans le notebook précédent (Régression Logistique). Le modèle construit précédemment atteignait 70 % de précision sur le jeu de test. Votre nouveau modèle devrait faire mieux !

**Énoncé du problème** : Vous disposez d'un jeu de données ("data.h5") contenant :
 - un ensemble d'entraînement de `m_train` images étiquetées chat (1) ou non-chat (0)
 - un ensemble de test de `m_test` images étiquetées chat et non-chat
 - chaque image est de forme (num_px, num_px, 3) où 3 correspond aux 3 canaux (RGB).

Chargez les données en exécutant la cellule ci-dessous.

In [ ]:
train_x_orig, train_y, test_x_orig, test_y, classes = load_data()

Le code suivant affiche une image du jeu de données. Vous pouvez modifier l'index et relancer la cellule pour explorer d'autres images.

In [ ]:
# Exemple d'image
index = 11
plt.imshow(train_x_orig[index])
print ("" + str(train_y[0,index]) + ". C'est une image " + classes[train_y[0,index]].decode("utf-8") + ".")
plt.show()

In [ ]:
# Explorer le jeu de données

m_train = train_x_orig.shape[0]
num_px = train_x_orig.shape[1]
m_test = test_x_orig.shape[0]

print ("Nombre d'exemples d'entraînement : " + str(m_train))
print ("Nombre d'exemples de test : " + str(m_test))
print ("Chaque image est de taille : (" + str(num_px) + ", " + str(num_px) + ", 3)")
print ("train_x_orig shape: " + str(train_x_orig.shape))
print ("train_y shape: " + str(train_y.shape))
print ("test_x_orig shape: " + str(test_x_orig.shape))
print ("test_y shape: " + str(test_y.shape))

Comme d'habitude, vous redimensionnez et normalisez les images avant de les transmettre au réseau. Le code est donné dans la cellule ci-dessous.

<img src="images/imvectorkiank.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>Figure 1</b> : Conversion d'une image en vecteur.</font></center></caption>

In [ ]:
# Redimensionner les exemples d'entraînement et de test
train_x_flatten = train_x_orig.reshape(train_x_orig.shape[0], -1).T # Le "-1" dans reshape permet d'aplatir les dimensions restantes
test_x_flatten = test_x_orig.reshape(test_x_orig.shape[0], -1).T

# Normaliser les données pour avoir des valeurs entre 0 et 1.
train_x = train_x_flatten/255
test_x = test_x_flatten/255.


print ("train_x's shape: " + str(train_x.shape))
print ("test_x's shape: " + str(test_x.shape))

**Note**:
$12\,288$ est égal à $64 \times 64 \times 3$, qui correspond à la taille d'un vecteur image aplati.

<a name='3'></a>
## 3 - Architecture du modèle

<a name='3-1'></a>
### 3.1 - Réseau de neurones à 2 couches

Maintenant que vous êtes familiarisé avec le jeu de données, il est temps de construire un réseau de neurones profond pour distinguer les images de chats des images sans chat.

Vous allez construire deux modèles différents :

- Un réseau de neurones à 2 couches
- Un réseau de neurones profond à L couches

Puis, vous comparerez les performances de ces modèles, et essaierez différentes valeurs pour $L$. 
Regardons les deux architectures :

<img src="images/2layerNN_kiank.png" style="width:650px;height:400px;">
<caption><center><font color='purple'><b>Figure 2</b> : Réseau de neurones à 2 couches. <br> Le modèle peut être résumé ainsi : ENTRÉE -> LINÉAIRE -> RELU -> LINÉAIRE -> SIGMOÏDE -> SORTIE.</font></center></caption>

<u><b>Architecture détaillée de la Figure 2</b></u> :
- L'entrée est une image (64,64,3) aplatie en un vecteur de taille $(12288,1)$. - Le vecteur correspondant $[x_0,x_1,...,x_{12287}]^T$ est multiplié par la matrice de poids $W^{[1]}$ de taille $(n^{[1]}, 12288)$.
- Ensuite, on ajoute un biais et on applique ReLU pour obtenir le vecteur $[a_0^{[1]}, a_1^{[1]},..., a_{n^{[1]}-1}^{[1]}]^T$.
- On multiplie le vecteur résultant par $W^{[2]}$ et on ajoute le biais. - Enfin, on applique la sigmoïde au résultat. Si le résultat est supérieur à 0,5, classez-le comme un chat.

<a name='3-2'></a>
### 3.2 - Réseau de neurones profond à L couches

Il est assez difficile de représenter un réseau de neurones profond à L couches avec la représentation ci-dessus. Voici néanmoins une représentation simplifiée :

<img src="images/LlayerNN_kiank.png" style="width:650px;height:400px;">
<caption><center><font color='purple'><b>Figure 3</b> : Réseau de neurones à L couches. <br> Le modèle peut être résumé ainsi : [LINÉAIRE -> RELU] $\times$ (L-1) -> LINÉAIRE -> SIGMOÏDE</font></center></caption>

<u><b>Architecture détaillée de la Figure 3</b></u> :
- L'entrée est une image (64,64,3) aplatie en un vecteur de taille (12288,1).
- Le vecteur correspondant $[x_0,x_1,...,x_{12287}]^T$ est multiplié par la matrice de poids $W^{[1]}$, puis on ajoute le biais $b^{[1]}$. Le résultat est appelé l'unité linéaire.
- Ensuite, on applique ReLU à l'unité linéaire. Ce processus peut être répété plusieurs fois pour chaque $(W^{[l]}, b^{[l]})$ selon l'architecture du modèle.
- Enfin, on applique la sigmoïde à la dernière unité linéaire. Si le résultat est supérieur à 0,5, on classifie l'image comme un chat.

<a name='3-3'></a>
### 3.3 - Méthodologie générale

Comme d'habitude, vous suivrez la méthodologie de l'apprentissage profond pour construire le modèle :

1. Initialiser les paramètres / Définir les hyperparamètres
2. Boucle sur num_iterations :
    a. Propagation avant
    b. Calculer la fonction de coût
    c. Rétropropagation
    d. Mettre à jour les paramètres (en utilisant les paramètres et les gradients de la rétropropagation)
3. Utiliser les paramètres entraînés pour prédire les étiquettes

Vous pouvez maintenant implémenter ces deux modèles !

<a name='4'></a>
## 4 - Réseau de neurones à deux couches

<a name='ex-1'></a>
### Exercice 1 - two_layer_model 
Utilisez les fonctions auxiliaires du notebook précédent pour construire un réseau de neurones à 2 couches avec la structure suivante : *LINÉAIRE -> RELU -> LINÉAIRE -> SIGMOÏDE*. Les fonctions et leurs entrées sont :
```python
def initialize_parameters(n_x, n_h, n_y):
 ...
 return paramètres 
def linear_activation_forward(A_prev, W, b, activation):
 ...
 return A, cache
def compute_cost(AL, Y):
 ...
 return coût
def linear_activation_backward(dA, cache, activation):
 ...
 return dA_prev, dW, db
def update_parameters(paramètres, grads, learning_rate):
 ...
 return paramètres
```

In [ ]:
### CONSTANTES DÉFINISSANT LE MODÈLE ####
n_x = 12288 # num_px * num_px * 3
n_h = 7
n_y = 1
layers_dims = n_x, n_h, n_y
learning_rate = 0.0075

In [ ]:
# FONCTION : two_layer_model
def two_layer_model(X, Y, layers_dims, learning_rate = 0.0075, num_iterations = 3000, print_cost=True):
    """
 Implémente un réseau de neurones à deux couches : LINÉAIRE->RELU->LINÉAIRE->SIGMOÏDE.
 
 Arguments :
 
 X -- données d'entrée, de forme (n_x, nombre d'exemples)
 Y -- vecteur d'étiquettes réelles (contenant 1 si chat, 0 sinon), de forme (1, nombre d'exemples)
 layers_dims -- dimensions des couches (n_x, n_h, n_y)
 num_iterations -- nombre d'itérations de la boucle d'optimisation
 learning_rate -- taux d'apprentissage de la règle de mise à jour par descente de gradient
 print_cost -- Si True, affiche le coût toutes les 100 itérations
 
 Retourne :
 paramètres -- un dictionnaire contenant W1, W2, b1 et b2
 """
    
    np.random.seed(1)
    grads = {}
    costs = []                               # pour suivre l'évolution du coût
    m = X.shape[1]                           # nombre d'exemples
    n_x, n_h, n_y = layers_dims
    
    # Initialiser le dictionnaire des paramètres en appelant une fonction déjà implémentée 
    #(≈ 1 ligne de code) 
    # paramètres = ... 
    # VOTRE CODE COMMENCE ICI 
    
    # VOTRE CODE SE TERMINE ICI 
    # 
    # Extraire W1, b1, W2 et b2 du dictionnaire paramètres. 
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
 
    # Boucle (descente de gradient) 
    for i in range(0, num_iterations):
        # Propagation avant : LINÉAIRE -> RELU -> LINÉAIRE -> SIGMOÏDE. Entrées : "X, W1, b1, W2, b2". Sorties : "A1, cache1, A2, cache2". 
        #(≈ 2 lignes de code) 
        #A1, cache1 = forward_propagation(X,parameters)
        #A2, cache2 = forward_propagation(A1,parameters)
        # VOTRE CODE COMMENCE ICI 
        
        
        # VOTRE CODE SE TERMINE ICI 
        
        # Calculer le coût #(≈ 1 ligne de code) 
        
        #cost = compute_cost(Prediction,Value)        
        # VOTRE CODE COMMENCE ICI 

        # VOTRE CODE SE TERMINE ICI 
        
        # Initialisation de la rétropropagation 
        # Rétropropagation. Entrées : "dA2, cache2, cache1". Sorties : "dA1, dW2, db2 ; aussi dA0 (non utilisé), dW1, db1". 
        #(≈ 2 lignes de code) 
        # dA1, dW2, db2 = ... 
        # dA0, dW1, db1 = ... 
        # VOTRE CODE COMMENCE ICI 
 
        # VOTRE CODE SE TERMINE ICI # Assigner grads['dW1'] à dW1, grads['db1'] à db1, grads['dW2'] à dW2, grads['db2'] à db2 
        grads['dW1'] = dW1 
        grads['db1'] = db1
        grads['dW2'] = dW2
        grads['db2'] = db2
        
        # Mettre à jour paramètres. #(environ 1 ligne de code) 
        # paramètres = ... 
        # VOTRE CODE COMMENCE ICI 
        
        # VOTRE CODE SE TERMINE ICI 
        # Récupérer W1, b1, W2, b2 depuis paramètres 
        
        W1 = parameters["W1"] 
        b1 = parameters["b1"]
        W2 = parameters["W2"]
        b2 = parameters["b2"]
        
        # Afficher le coût toutes les 100 itérations 
        if print_cost and i % 100 == 0 or i == num_iterations - 1: 
            print("Coût après l'itération {}: {}".format(i, np.squeeze(coût)))
        if i % 100 == 0 or i == num_iterations:
            costs.append(cost)
    return parameters, costs

def plot_costs(costs, learning_rate=0.0075):
    plt.plot(np.squeeze(costs))
    plt.ylabel('cost')
    plt.xlabel('itérations (par centaines)')
    plt.title("Taux d'apprentissage =" + str(learning_rate))
    plt.show()

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2, print_cost=False)

print("Cost after first iteration: " + str(costs[0]))

two_layer_model_test(two_layer_model)

**Sortie attendue :**
```
Le coût après l'itération 1 doit être environ 0.69
```

<a name='4-1'></a>
### 4.1 - Entraîner le modèle 
Si votre code a réussi la cellule précédente, exécutez la cellule ci-dessous pour entraîner vos paramètres. 
- Le coût devrait diminuer à chaque itération. 
- Cela peut prendre jusqu'à 5 minutes pour exécuter 2500 itérations.

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2500, print_cost=True)
plot_costs(costs, learning_rate)

**Sortie attendue :**
<table>  <tr>
 <td> <b>Coût après l'itération 0</b></td>
 <td> 0.6930497356599888 </td>
 </tr>
 <tr>
 <td> <b>Coût après l'itération 100</b></td>
 <td> 0.6464320953428849 </td>
 </tr>
 <tr>
 <td> <b>...</b></td>
 <td> ... </td>
 </tr>
 <tr>
 <td> <b>Coût après l'itération 2499</b></td>
 <td> 0.04421498215868956 </td>
 </tr>
</table>

**Bien joué !** Vous avez entraîné le modèle avec succès. Grâce à l'implémentation vectorisée, l'entraînement est bien plus rapide.

Vous pouvez maintenant utiliser les paramètres entraînés pour classer les images du jeu de données. Exécutez la cellule ci-dessous pour voir les prédictions sur les ensembles d'entraînement et de test.

In [ ]:
predictions_train = predict(train_x, train_y, parameters)

**Sortie attendue :**
<table>  <tr>
 <td> <b>Précision</b></td>
 <td> 0.9999999999999998 </td>
 </tr>
</table>

In [ ]:
predictions_test = predict(test_x, test_y, parameters)

**Sortie attendue :**
<table>  <tr>
 <td> <b>Précision</b></td>
 <td> 0.72 </td>
 </tr>
</table>

### Bien joué ! Il semble que votre réseau de neurones à 2 couches soit plus performant (72 %) que l'implémentation de régression logistique (70 %). Voyons si vous pouvez faire encore mieux avec un modèle à $L$ couches.

**Note** : Vous remarquerez peut-être qu'entraîner le modèle sur moins d'itérations (par exemple 1500) donne une meilleure précision sur l'ensemble de test. C'est ce qu'on appelle l'**arrêt précoce** (*early stopping*), une technique pour éviter le surapprentissage.

<a name='5'></a>
## 5 - Réseau de neurones profond à L couches

<a name='ex-2'></a>
### Exercice 2 - L_layer_model 
Utilisez les fonctions auxiliaires implémentées précédemment pour construire un réseau de neurones à $L$ couches avec la structure suivante : *[LINÉAIRE -> RELU]$\times$(L-1) -> LINÉAIRE -> SIGMOÏDE*. Les fonctions et leurs entrées sont :
```python
def initialize_parameters_deep(layers_dims):
 ...
 return paramètres 
def L_model_forward(X, paramètres):
 ...
 return AL, caches
def compute_cost(AL, Y):
 ...
 return coût
def L_model_backward(AL, Y, caches):
 ...
 return grads
def update_parameters(paramètres, grads, learning_rate):
 ...
 return paramètres
```

In [ ]:
### CONSTANTS ###
layers_dims = [12288, 20, 7, 5, 1] # modèle à 4-couche

In [ ]:
# FONCTION : L_layer_model
def L_layer_model(X, Y, layers_dims, learning_rate = 0.0075, num_iterations = 3000, print_cost=False):
    """
 Implémente un réseau de neurones à L couches : [LINÉAIRE->RELU]*(L-1)->LINÉAIRE->SIGMOÏDE.
 
 Arguments :
 X -- données d'entrée, de forme (n_x, nombre d'exemples)
 Y -- vecteur d'étiquettes réelles (contenant 1 si chat, 0 sinon), de forme (1, nombre d'exemples)
 layers_dims -- liste contenant la taille d'entrée et la taille de chaque couche, de longueur (nombre de couches + 1).
 learning_rate -- taux d'apprentissage de la règle de mise à jour par descente de gradient
 num_iterations -- nombre d'itérations de la boucle d'optimisation
 print_cost -- si True, affiche le coût toutes les 100 étapes
 
 Retourne :
 paramètres -- paramètres appris par le modèle. Ils peuvent ensuite être utilisés pour prédire.
 """

    np.random.seed(1)
    costs = []                         # suivre l'évolution du coût
    
    # Initialisation des paramètres. 
    #(≈ 1 ligne de code) 
    # paramètres = ... 
    # VOTRE CODE COMMENCE ICI 

    # VOTRE CODE SE TERMINE ICI 
    # Boucle (descente de gradient) 
    for i in range(0, num_iterations):# Propagation avant : [LINÉAIRE -> RELU]*(L-1) -> LINÉAIRE -> SIGMOÏDE. 
        #(≈ 1 ligne de code) 
        # AL, caches = ... 
        # VOTRE CODE COMMENCE ICI 

        # VOTRE CODE SE TERMINE ICI 
        # Calculer le coût. 
        #(≈ 1 ligne de code) 
        # cost = ... 
        # VOTRE CODE COMMENCE ICI 

        # VOTRE CODE SE TERMINE ICI 
        # Rétropropagation. 
        #(≈ 1 ligne de code) 
        # grads = ... 
        # VOTRE CODE COMMENCE ICI 

        # VOTRE CODE SE TERMINE ICI 
        # Mettre à jour paramètres. 
        #(≈ 1 ligne de code) 
        # paramètres = ... 
        # VOTRE CODE COMMENCE ICI 
        
        # VOTRE CODE SE TERMINE ICI 
        # Afficher le coût toutes les 100 itérations 
        if print_cost and i % 100 == 0 or i == num_iterations - 1: 
            print("Coût après l'itération {}: {}".format(i, np.squeeze(coût)))
        if i % 100 == 0 or i == num_iterations:
            costs.append(cost)
    
    return parameters, costs

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 1, print_cost = False)

print("Cost after first iteration: " + str(costs[0]))

L_layer_model_test(L_layer_model)

<a name='5-1'></a>
### 5.1 - Entraîner le modèle 
Si votre code a réussi la cellule précédente, exécutez la cellule ci-dessous pour entraîner votre modèle en tant que réseau de neurones à 4 couches. 
- Le coût devrait diminuer à chaque itération. 
- Cela peut prendre jusqu'à 5 minutes pour exécuter 2500 itérations.

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 2500, print_cost = True)

**Sortie attendue :**
<table>  <tr>
 <td> <b>Coût après l'itération 0</b></td>
 <td> 0.771749 </td>
 </tr>
 <tr>
 <td> <b>Coût après l'itération 100</b></td>
 <td> 0.672053 </td>
 </tr>
 <tr>
 <td> <b>...</b></td>
 <td> ... </td>
 </tr>
 <tr>
 <td> <b>Coût après l'itération 2499</b></td>
 <td> 0.088439 </td>
 </tr>
</table>

In [ ]:
pred_train = predict(train_x, train_y, parameters)

**Sortie attendue :**
<table>
 <tr>
 <td>
 <b>Précision entraînement</b>
 </td>
 <td>
 0.985645933014
 </td>
 </tr>
</table>

In [ ]:
pred_test = predict(test_x, test_y, parameters)

**Sortie attendue :**
<table>  <tr>
 <td> <b>Précision sur l'ensemble de test</b></td>
 <td> 0.8 </td>
 </tr>
</table>

### Félicitations ! Il semble que votre réseau à 4 couches soit plus performant (80 %) que votre réseau à 2 couches (72 %) sur le même ensemble de test.

C'est une très bonne performance pour cette tâche. Excellent travail !

<a name='6'></a>
## 6 - Analyse des résultats

Commencez par observer des images mal classées par le modèle à L couches. Cela montre quelques exemples d'erreurs.

In [ ]:
print_mislabeled_images(classes, test_x, test_y, pred_test)

**Quelques types d'images que le modèle a tendance à mal classer :**
- Corps de chat dans une position inhabituelle
- Chat sur un fond de couleur similaire
- Couleur ou espèce de chat inhabituelle
- Angle de la caméra
- Luminosité de l'image
- Variation d'échelle (chat très grand ou très petit dans l'image)

### Félicitations !

Vous venez de construire et d'entraîner un réseau de neurones profond à L couches, et de l'appliquer pour distinguer les chats des non-chats.

Si vous le souhaitez, vous pouvez tester votre modèle avec votre propre image dans l'exercice optionnel ci-dessous.

<a name='7'></a>
## 7 - Testez avec votre propre image (exercice optionnel)

Pour tester le modèle avec votre propre image, suivez ces étapes :

1. Ajoutez votre image dans le dossier `images/` de ce répertoire.
2. Modifiez le nom de l'image dans le code ci-dessous.
3. Exécutez la cellule et vérifiez si l'algorithme a raison (1 = chat, 0 = non-chat) !

In [ ]:
## DÉBUT DU CODE ##
my_image = "my_image.jpg" 
# modifiez avec le nom de votre fichier image
my_label_y = [1] # la vraie classe de votre image (1 -> chat, 0 -> non-chat)
## FIN DU CODE ##
fname = "images/" + my_image
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T

my_predicted_image = predict(image, my_label_y, parameters)


print ("y = " + str(np.squeeze(my_predicted_image)) + ", votre modèle à L couches prédit une image \"" + classes[int(np.squeeze(my_predicted_image)),].decode("utf-8") +  "\".")

**Références** :

- Pour le rechargement automatique des modules externes : http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
- Deep Learning Specialization DeepLearning.ai sur Coursera